In [36]:
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

def add_user_message(messages, text):
    user_message = {"role":"user", "content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant", "content":text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):  
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"]=stop_sequences

    message = client.messages.create(**params)
    
    return message.content[0].text

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python" or "json" or "regex",
    "solution_criteria": "Define and elaborate what good solution would look like. E.g. For AWS Lambda configuration you can say the solution must include proper runtime check, memory size, timeout as the basic structure... and so on. Be very prescriptive and precise when you explain this."
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages=[]
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [37]:
dataset = generate_dataset()

with open("dataset.json","w") as f:
    json.dump(dataset, f, indent=2)


In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
                Please solve the following task:

                {test_case["task"]}

                * Respond only with Python, JSON or plain Regex
                * Do not add any comments, commentary or explanations 
                """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [45]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Solution Criteria:
<solution>
{test_case["solution_criteria"]}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [26]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [49]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_grade = grade_syntax(output, test_case)
    
    score = (model_score + syntax_grade) / 2
    
    print(f"Your average score is: {score}")
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [50]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score =mean([result["score"] for result in results]) 
    print(f"Average score: {average_score}")
    
    return results

In [51]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)
print(json.dumps(results, indent=2))

Your average score is: 8.5
Your average score is: 7.5
Your average score is: 8.0
Average score: 8.0
[
  {
    "output": "\nimport re\nimport ipaddress\n\ndef validate_s3_bucket_name(bucket_name: str) -> bool:\n    if bucket_name is None or not isinstance(bucket_name, str):\n        return False\n    \n    if len(bucket_name) < 3 or len(bucket_name) > 63:\n        return False\n    \n    if not re.match(r'^[a-z0-9][a-z0-9.-]*[a-z0-9]$', bucket_name) and not re.match(r'^[a-z0-9]$', bucket_name):\n        return False\n    \n    if bucket_name.startswith('-') or bucket_name.endswith('-'):\n        return False\n    \n    if bucket_name.startswith('.') or bucket_name.endswith('.'):\n        return False\n    \n    if '..' in bucket_name:\n        return False\n    \n    try:\n        ipaddress.ip_address(bucket_name)\n        return False\n    except ValueError:\n        pass\n    \n    return True\n",
    "test_case": {
      "task": "Create a Python function that validates an AWS S3 buck